# Market Basket and Cross-Sell Analysis

This notebook analyses grocery transaction data to identify product affinity patterns, promotional halo effects, customer purchase segments, and simple basket-based recommendation opportunities. It has been cleaned for portfolio use; course submission details and assessment-specific metadata have been removed.

## Analysis scope
- Construct basket-level product combinations
- Calculate support, confidence, and lift for product pairs
- Analyse category-level affinities and anchor categories
- Estimate promotional halo effects for high-affinity pairs
- Segment households using K-Means based on category purchase behaviour
- Build a simple recommendation prototype using lift and confidence


# 1. Market Basket Construction


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

In [ ]:
transactions = pd.read_csv('data/csv/transaction_data.csv')
products = pd.read_csv('data/csv/product.csv')
causal_data = pd.read_csv('data/csv/causal_data.csv')

In [ ]:
transactions.head()

In [ ]:
products.head()

In [ ]:
# clean the data
transactions.dropna()
products.dropna()

In [ ]:
# 1-1. Group transaction_data by basket_id to identify complete shopping baskets
basket_groups = transactions.groupby('BASKET_ID')
basket_groups.head()

In [ ]:
# 1-2. For each basket, list all products purchased together
#avoid duplicate product
transactions_nodup = transactions.drop_duplicates(subset=["BASKET_ID", "PRODUCT_ID"])

#list products being purchased together based on BASKET_ID
basket_groups = transactions_nodup.groupby('BASKET_ID')['PRODUCT_ID'].agg(list).reset_index(name= "PROD_PURCHASE_TOGETHER")

#count basket sizes
basket_groups['BASKET_SIZE'] = basket_groups['PROD_PURCHASE_TOGETHER'].apply(len)
basket_groups.head(8)


In [ ]:
basket_exploded = basket_groups.explode("PROD_PURCHASE_TOGETHER").rename(columns={
    "PROD_PURCHASE_TOGETHER": "PRODUCT_ID"
})


basket_exploded.head(8)

In [ ]:
# 1-3. Merge with product to get product details (department, commodity, brand)
# explode the data to merge so that it can merge with products
basket_exploded = basket_groups.explode("PROD_PURCHASE_TOGETHER").rename(columns={
    "PROD_PURCHASE_TOGETHER": "PRODUCT_ID"
})


basket_summary = basket_exploded.merge(
    products[['PRODUCT_ID','DEPARTMENT', 'BRAND', 'COMMODITY_DESC']], 
    on= 'PRODUCT_ID',
    how= 'left'
)

basket_summary.head(10)


In [ ]:
# 1-4. Filter to focus on meaningful basket sizes (e.g., 3+ items) to avoid noise
filtered_basket = basket_summary[basket_summary['BASKET_SIZE'] >= 3]
filtered_basket.head(10)


## why BASKET_SIZE' >= 3
* Very small baskets (1–2 items) rarely contain meaningful co-occurrence patterns and introduce noise.
* Therefore, baskets with fewer than 3 items were removed following common market-basket-analysis practices.

# 2. Product Affinity Metrics

In [ ]:
# The number of times the product appears in different baskets.
n_prod_appear_in_basket = filtered_basket.groupby('PRODUCT_ID')["BASKET_ID"].nunique().reset_index(name= "N_in_BASKET")

n_prod_appear_in_basket.head(20)


In [ ]:
# Generate all possible product pairs within each basket with self-join
pairs = filtered_basket.merge(
    filtered_basket,
    on="BASKET_ID",
    suffixes=("_A", "_B")
)


# remove when a product form a pair with itself 
pairs = pairs[pairs["PRODUCT_ID_A"] != pairs["PRODUCT_ID_B"]]
# avoid duplicate pairs 
pairs = pairs[pairs["PRODUCT_ID_A"] < pairs["PRODUCT_ID_B"]]

# COUNT_AB: the times each pair has appeared in different baskets 計算 COUNT_AB（多少個不同籃子有這個 pair）
pair_counts = (
    pairs.groupby(["PRODUCT_ID_A", "PRODUCT_ID_B"])["BASKET_ID"]
    .nunique()
    .reset_index(name="COUNT_AB")
)

# add COUNT_A & COUNT_B, the times each products has appeared in different baskets respectively 
pair_counts = (
    pair_counts
    .merge(
        n_prod_appear_in_basket.rename(columns={"PRODUCT_ID": "PRODUCT_ID_A", "N_in_BASKET": "COUNT_A"}),
        on="PRODUCT_ID_A",
        how="left"
    )
    .merge(
        n_prod_appear_in_basket.rename(columns={"PRODUCT_ID": "PRODUCT_ID_B", "N_in_BASKET": "COUNT_B"}),
        on="PRODUCT_ID_B",
        how="left"
    )
)

pair_counts.head(10)


In [ ]:
# support
# How often does pair appear together (% of baskets containing both)
pair_counts['Support'] = pair_counts['COUNT_AB'] / len(basket_groups)
pair_counts

In [ ]:
# confidenct
# If product A is purchased, probability of product B being purchased
# only consider P(B|A) here due to the requirement from the assignment
pair_counts['Confidence_A_B'] = pair_counts['COUNT_AB'] / pair_counts['COUNT_A']
pair_counts


In [ ]:
# Lift
# How much more likely B is purchased given A vs. baseline probability of B
# whether the proportion of purchasing B after purchasing A higher than usual
pair_counts['Lift'] = pair_counts['COUNT_AB'] * len(basket_groups) / (pair_counts['COUNT_A'] * pair_counts['COUNT_B'])
pair_counts

In [ ]:
# 2-2. Focus on high-lift product pairs
# to avoid noise rule, minimum thresholds are setted
pair_filtered = pair_counts[
    (pair_counts["COUNT_AB"] >= 3) & # this rule at least happened 3 times
    (pair_counts["COUNT_A"] >= 10) & # this rule at least happened 10 times
    (pair_counts["COUNT_B"] >= 10)   # this rule at least happened 10 times
]

min_transaction = 10 #at least 10 transactions contain this product pair
min_support = min_transaction / len(basket_groups)

min_lift =1.2 # set threshold for lift
high_lift_pairs = pair_filtered[(pair_filtered['Lift'] >= min_lift) & 
                                (pair_filtered['Support'] >= min_support)]
high_lift_pairs = high_lift_pairs.sort_values('Lift', ascending= False)

high_lift_pairs = high_lift_pairs.merge(
    products[['PRODUCT_ID', 'COMMODITY_DESC']],
    left_on='PRODUCT_ID_A',
    right_on='PRODUCT_ID',
    how='left'
).rename(columns={'COMMODITY_DESC': 'A_DESC'}).drop(columns=['PRODUCT_ID'])

high_lift_pairs = high_lift_pairs.merge(
    products[['PRODUCT_ID', 'COMMODITY_DESC']],
    left_on='PRODUCT_ID_B',
    right_on='PRODUCT_ID',
    how='left'
).rename(columns={'COMMODITY_DESC': 'B_DESC'}).drop(columns=['PRODUCT_ID'])


# Top 50 product pairs with highest cross-selling potential (with lift metrics)
high_lift_pairs.head(50)

top_50_high_lift_products = high_lift_pairs[['PRODUCT_ID_A', 'PRODUCT_ID_B', 
                                             'A_DESC', 'B_DESC', 
                                             'Support', 'Confidence_A_B', 'Lift']].head(50)

top_50_high_lift_products


# 3. Category-Level Affinity Analysis

In [ ]:
# 3-1.Aggregate affinity patterns to department/commodity level

# ensure the name is neat
products['COMMODITY_DESC'] = products['COMMODITY_DESC'].str.strip()
products['DEPARTMENT'] = products['DEPARTMENT'].str.strip()

# combine PRODUCT_A's information
commodity_pairs = pair_filtered.merge(
    products[['PRODUCT_ID' , 'COMMODITY_DESC' , 'DEPARTMENT']],
    left_on= 'PRODUCT_ID_A',
    right_on= 'PRODUCT_ID',
    how= 'left'
).rename(columns= {'COMMODITY_DESC': 'COMMODITY_A', 'DEPARTMENT': 'DEPARTMENT_A'}).drop(columns=['PRODUCT_ID'])

# combine PRODUCT_B's information
commodity_pairs = commodity_pairs.merge(
    products[['PRODUCT_ID' , 'COMMODITY_DESC' , 'DEPARTMENT']],
    left_on= 'PRODUCT_ID_B',
    right_on= 'PRODUCT_ID',
    how= 'left'
).rename(columns= {'COMMODITY_DESC': 'COMMODITY_B', 'DEPARTMENT': 'DEPARTMENT_B'}).drop(columns=['PRODUCT_ID'])

# deal with invaild value
invalid_categories = ['', np.nan, None]

commodity_pairs['COMMODITY_A'] = commodity_pairs['COMMODITY_A'].replace(invalid_categories, np.nan)
commodity_pairs['COMMODITY_B'] = commodity_pairs['COMMODITY_B'].replace(invalid_categories, np.nan)
category_pairs = commodity_pairs.dropna(subset=['COMMODITY_A', 'COMMODITY_B'])

commodity_pairs

In [ ]:
# create a new frame only about categories 
category_level_pairs = commodity_pairs.groupby(['COMMODITY_A','COMMODITY_B'])['COUNT_AB'].sum().reset_index(name= 'COUNT_AB_sum')

unique_filtered_baskets = (filtered_basket.groupby('COMMODITY_DESC')['BASKET_ID'].nunique().reset_index(name= 'n_in_BASKET_category'))

# merge category-information
category_level_pairs = category_level_pairs.merge(
    unique_filtered_baskets.rename(columns= {'COMMODITY_DESC' : 'COMMODITY_A'}), 
    on='COMMODITY_A', 
    how='left'
).rename(columns= {'n_in_BASKET_category' : 'n_in_BASKET_category_A'})

category_level_pairs = category_level_pairs.merge(
    unique_filtered_baskets.rename(columns= {'COMMODITY_DESC' : 'COMMODITY_B'}), 
    on='COMMODITY_B', 
    how='left'
).rename(columns= {'n_in_BASKET_category' : 'n_in_BASKET_category_B'})

category_level_pairs

In [ ]:
# calculate support, confidence and lift in category level
category_level_pairs['Support_category'] = category_level_pairs['COUNT_AB_sum'] / len(basket_groups)
category_level_pairs['Confidence_category'] = category_level_pairs['COUNT_AB_sum'] / category_level_pairs['n_in_BASKET_category_A']
category_level_pairs['Lift_category'] = category_level_pairs['Confidence_category'] / (category_level_pairs['n_in_BASKET_category_B'] / len(basket_groups))

category_level_pairs

In [ ]:
# 3-2. Create affinity heatmaps showing which categories drive cross-selling
# visualization 

# there are "BABYFOOD", "BABY FOOD" and "BABY FOODS" in the data but they represent the same things
normalize_map = {
    "BABYFOOD": "BABY FOODS",
    "BABY FOOD": "BABY FOODS"
}

# Apply normalization BEFORE pivot
category_level_pairs['COMMODITY_A'] = category_level_pairs['COMMODITY_A'].replace(normalize_map)
category_level_pairs['COMMODITY_B'] = category_level_pairs['COMMODITY_B'].replace(normalize_map)


# remove those pair with themselves
category_level_pairs = category_level_pairs[
    category_level_pairs['COMMODITY_A'] != category_level_pairs['COMMODITY_B']].copy()

# set threshold
min_category_support = 0.0001
min_category_lift = 1.2

# filtered
category_affinities_filtered = category_level_pairs[
    (category_level_pairs['Support_category'] >= min_category_support) &
    (category_level_pairs['Lift_category'] >= min_category_lift)
].copy()

category_affinities_filtered = category_affinities_filtered.sort_values("Lift_category", ascending=False)


# pivot table
category_pivot = category_affinities_filtered.pivot_table(
    index='COMMODITY_A', 
    columns='COMMODITY_B', 
    values='Lift_category', 
    fill_value=1.0 # Lift=1.0 means irrelevant
)


# visualize category_pivot with heat map
plt.figure(figsize=(20, 18)) 
sns.heatmap(
    category_pivot, 
    annot=False, 
    cmap="YlOrRd", 
    fmt=".2f", 
    linewidths=.5, 
    cbar_kws={'label': 'Lift Score (Category-Level)'}
)
plt.title('Category Affinity Heatmap (Lift Score) - Filtered')
plt.xlabel('COMMODITY B')
plt.ylabel('COMMODITY A')

plt.show()


In [ ]:
# Top 15% Categories Affinity Heatmap
# the heatmap 'Category Affinity Heatmap (Lift Score) - Filtered' is too large to read
# Find those top category-pair 
top_category = category_affinities_filtered.head(35)

# pivot table
top_category_pivot = top_category.pivot_table(
    index='COMMODITY_A', 
    columns='COMMODITY_B', 
    values='Lift_category', 
    fill_value=1.0 # Lift=1.0 
)


# visualize category_pivot with heat map
plt.figure(figsize=(20, 18)) 
sns.heatmap(
    top_category_pivot, 
    annot=False, 
    cmap="YlOrRd", 
    fmt=".2f", 
    linewidths=.5, 
    cbar_kws={'label': 'Lift Score (Category-Level)'}
)
plt.title('Category Affinity Heatmap (Lift Score) - Top')
plt.xlabel('COMMODITY B')
plt.ylabel('COMMODITY A')

plt.show()


In [ ]:
category_affinities_filtered

In [ ]:
# 3-3.Identify “anchor categories” that generate high cross-category purchases
# this means to find pairs with high confidence

# remove when when A and B belongs to same category 
cross_only = category_affinities_filtered[
    category_affinities_filtered['COMMODITY_A'] != category_affinities_filtered['COMMODITY_B']
]

anchor_categories = (category_affinities_filtered.groupby('COMMODITY_A')['Confidence_category']
                     .mean()
                     .sort_values(ascending= False))

total_categories = len(anchor_categories)
cofidence_category_15 = total_categories * 0.15
cofidence_category_15 #34.9999 ~ 35

# top 15% 
anchor_categories.head(35)


In [ ]:
# 「Top 15% Categories」Affinity Heatmap (easier to read)
top_cats = anchor_categories.head(35).index
subset = category_affinities_filtered[
    category_affinities_filtered['COMMODITY_A'].isin(top_cats) &
    category_affinities_filtered['COMMODITY_B'].isin(top_cats)
]

pivot = subset.pivot(index='COMMODITY_A', columns='COMMODITY_B', values='Lift_category')

plt.figure(figsize=(12,10))
sns.heatmap(pivot, cmap="YlOrRd", annot=False)
plt.title("Top 20 Anchor Categories – Cross-Category Lift Heatmap")
plt.show()

### **3-4. Spot unexpected affinities that suggest opportunity**
find those pairs that we did not expect they will appear together

In addition to intuitive combinations (e.g., Baby Foods ↔ Infant Formula), several high-lift pairs do not belong to the same shopping mission. These represent **unexpected affinities** that reveal deeper behavioural patterns beyond straightforward category links.

### **1. Baby Foods → Cheeses / Meat – Shelf Stable / Powder Drink Mix**

**Why unexpected:**
Baby Food is part of an infant-care mission, while cheeses, shelf-stable meats, and powdered drink mixes belong to adult cooking or beverage replenishment.

**Insight:**
This pattern suggests *young-family pantry stock-ups*, where households with children replenish both baby items and general pantry goods in the same trip.
High lift strengthens its value for cross-category promotions.

---

### **2. Convenient Breakfast Snacks → Meat – Shelf Stable / Powder Drink Mix**

**Why unexpected:**
Breakfast bars and wholesome snacks do not naturally pair with shelf-stable meats or powdered drinks.

**Insight:**
It reflects a *convenience-driven replenishment mission*, where shoppers who prioritise quick meals tend to stock multiple fast-prep items across departments in one visit.

---

### **3. Cookies / Cones → Powder Drink Mix**

**Why unexpected:**
Sweet snacks and powdered beverages do not share a clear consumption context.

**Insight:**
This pattern likely arises from *family bulk purchasing*, where households restock snacks and drink mixes together as part of a routine stock-up cycle.

---

### **4. Frozen Meat / Meat Dinners → Powder Drink Mix**

**Why unexpected:**
Frozen entrées and powdered beverages belong to completely different sections and missions.

**Insight:**
The affinity indicates that some households conduct **broad stock-up trips** that span both frozen meals and drink mixes, revealing a cross-department replenishment behaviour useful for bundle design.

---

### **Summary Insight**

Across these pairs, the common theme is **multi-department stock-up behaviour**, rather than single-mission shopping.
Although each lift value is moderate, the breadth of connections makes these categories strong candidates for cross-category promotions, bundle offers, and personalised recommendations.




# 4. Promotional Impact on Affinities



In [ ]:
# 4-1. Join with causal_data to identify promoted products
#  merge causal_data with transactions
promo_transactions = transactions.merge(
    causal_data,
    on= ['PRODUCT_ID', 'WEEK_NO'],
    how= 'left'
)
promo_transactions = promo_transactions.drop(columns=['STORE_ID_y'])
promo_transactions['display'] = promo_transactions['display'].fillna(0)
promo_transactions['mailer'] = promo_transactions['mailer'].fillna(0)
promo_transactions

In [ ]:
# identify promoted products
promo_transactions['display_promoted'] = promo_transactions['display'] != 0
promo_transactions['mailer_promoted'] = promo_transactions['mailer'] != 0

promo_transactions['is_promoted'] = promo_transactions['display_promoted'] | promo_transactions['mailer_promoted']
promo_transactions


In [ ]:
# 4-2. Analyse if promotions on “anchor products” increase purchases of high-affinity partners
# 4-3. Calculate "halo effect" - incremental revenue from affinity products when anchor is promoted

# --- 4.2.1 Define the halo effect calculation function ---
def calculate_halo_effect(promo_transactions_df, ID_anchor, ID_partner):
# Quantify the incremental sales lift (Halo Effect) that promotion of product ID_A generates for its partner product ID_B.
# Causal logic: Promoting ID_A (cause) → increases sales of ID_B (effect).

# id_a: Anchor product (the promoted item)
# id_b: Partner product (the item receiving the halo impact)

# Meaning: This simulates an A/B test by comparing ID_B’s sales between the “anchor-promoted group” and the “anchor-not-promoted control group.”

    # Identify baskets that purchased A (anchor opportunities)
    # 1. Filter baskets that contain the anchor product A
    # Meaning: We analyse only customers who had the opportunity to see A’s promotion and decide whether to purchase B.
    baskets_with_A = promo_transactions_df[promo_transactions_df['PRODUCT_ID'] == ID_anchor]['BASKET_ID'].unique()

    # 2. Filter relevant transaction rows: retrieve all transaction records within these baskets
    target_transactions = promo_transactions_df[promo_transactions_df['BASKET_ID'].isin(baskets_with_A)].copy()
    
    if len(target_transactions) == 0:
        return np.nan, np.nan, np.nan
        
    # 3. Aggregate: compute each basket's promotion status and B’s sales

    # Determine whether A was promoted in those baskets
    # Create a promotion flag for anchor product A (boolean mask)
    # Meaning: Identify rows in target_transactions where the item is A AND it was promoted.
    # A_PROMOTED (grouping key): indicates whether A was under promotion when purchased in that basket.
    target_transactions['A_IS_PROMOTED_FLAG'] = (
        (target_transactions['PRODUCT_ID'] == ID_anchor) & 
        (target_transactions['is_promoted'])
    )

    # Compute B’s spend for each basket.
    # B. Create revenue measure for partner product B (numeric)
    # Meaning: Only rows corresponding to product B contribute to its sales; all others are assigned 0.
    # This captures how much revenue each transaction contributes for partner product B.
    target_transactions['B_REVENUE_LINE'] = np.where(
        target_transactions['PRODUCT_ID'] == ID_partner, 
        target_transactions['SALES_VALUE'], 
        0
    )

    # Aggregate at the basket level (grouped by BASKET_ID)
    basket_analysis = target_transactions.groupby('BASKET_ID').agg(
        
        # A_PROMOTED: A basket is marked True if any row has A_IS_PROMOTED_FLAG = True
        A_PROMOTED=('A_IS_PROMOTED_FLAG', 'max'), 

        # B_REVENUE: total revenue generated by partner product B within the basket
        B_REVENUE=('B_REVENUE_LINE', 'sum') 
    ).reset_index()


    # 4. Group comparison (simulating Treatment vs. Control in an A/B test)
    # Create the promoted (treatment) and non-promoted (control) groups
    promo_group = basket_analysis[basket_analysis['A_PROMOTED'] == True]
    control_group = basket_analysis[basket_analysis['A_PROMOTED'] == False]
    
    

    # 5. Compute average revenue (Analyse whether promotions increase B’s purchases)
    # Meaning: Compare the average B revenue between the two groups.
    promo_revenue = promo_group['B_REVENUE'].mean()
    control_revenue = control_group['B_REVENUE'].mean()
    
    # 6. Compute the Halo Effect
    # Halo Effect: the net incremental lift in B’s revenue driven by promoting A
    halo_effect_per_basket = promo_revenue - control_revenue # Incremental revenue per basket

    total_halo_effect_revenue = halo_effect_per_basket * len(promo_group) # Total incremental revenue
    
    return halo_effect_per_basket, total_halo_effect_revenue, len(promo_group)


# --- 4.2.2 Batch execution for the Top 50 rules ---

N_RULES_TO_TEST = 50 
test_rules = high_lift_pairs.head(N_RULES_TO_TEST).copy() # Select the top 50 rules for testing

# Initialize result columns
test_rules['Halo_Effect_Per_Basket'] = np.nan
test_rules['Total_Halo_Effect_Revenue'] = np.nan
test_rules['Promo_Basket_Count'] = np.nan


# Iterate through each rule and compute metrics
for index, row in test_rules.iterrows():
    ID_anchor = row['PRODUCT_ID_A']
    ID_partner = row['PRODUCT_ID_B']
    
    halo_per_basket, total_halo, promo_count = calculate_halo_effect(promo_transactions, ID_anchor, ID_partner)
    
    test_rules.loc[index, 'Halo_Effect_Per_Basket'] = halo_per_basket
    test_rules.loc[index, 'Total_Halo_Effect_Revenue'] = total_halo
    test_rules.loc[index, 'Promo_Basket_Count'] = promo_count

# Output results: rank by financial impact for final strategy recommendations
final_halo_recommendation = test_rules.sort_values(
    'Total_Halo_Effect_Revenue', 
    ascending=False
)



final_halo_recommendation.head()

In [ ]:
# Top 20 Halo Effect bar chart
# to better understand the pairs that brings increase on sales
top20 = final_halo_recommendation.head(20)

plt.figure(figsize=(12,8))
sns.barplot(
    data=top20,
    x="Total_Halo_Effect_Revenue",
    y=top20['PRODUCT_ID_A'].astype(str) + " → " + top20['PRODUCT_ID_B'].astype(str),
    color= "#8B5151"
)
plt.title("Top 20 Product Pairs by Promotional Halo Effect")
plt.xlabel("Incremental Revenue")
plt.ylabel("Product Pair")
plt.show()


In [ ]:
# 4-4. Test if displaying affinity products together increases combined purchase probability
# Use the rule with the highest Halo Effect for this test (TEST_ID_A and TEST_ID_B defined above)
# Ensure these ID variables are available
TEST_ID_A = final_halo_recommendation.iloc[0]['PRODUCT_ID_A'] 
TEST_ID_B = final_halo_recommendation.iloc[0]['PRODUCT_ID_B']


# 1. Filter transactions containing only products A and B
# Meaning: Define the analysis scope by isolating transactions involving these two products.
AB_transactions = promo_transactions[
    (promo_transactions['PRODUCT_ID'] == TEST_ID_A) | (promo_transactions['PRODUCT_ID'] == TEST_ID_B)
].copy()


AB_transactions

In [ ]:
# 2. Create row-level flags 
# Meaning: These serve as the foundation for aggregation, converting complex logical conditions into 0/1 indicators.

# Flag A: whether the row corresponds to product A
AB_transactions['IS_A'] = (AB_transactions['PRODUCT_ID'] == TEST_ID_A)
# Flag B: whether the row corresponds to product B
AB_transactions['IS_B'] = (AB_transactions['PRODUCT_ID'] == TEST_ID_B)
# Flag P_A: whether the row is product A under promotion
AB_transactions['IS_PROMO_A'] = AB_transactions['IS_A'] & AB_transactions['display_promoted']
# Flag P_B: whether the row is product B under promotion
AB_transactions['IS_PROMO_B'] = AB_transactions['IS_B'] & AB_transactions['display_promoted']

AB_transactions

In [ ]:
# 3. Basket-level aggregation (use 'max' for OR logic and 'sum' for counts)
# Meaning: Convert row-level True/False flags into basket-level indicators 
# (a basket is True if any row within it is True).

basket_status = AB_transactions.groupby('BASKET_ID').agg(
    
    # IS_A_PURCHASED: Whether the basket purchased A  (max: True if any row has IS_A = True)
    IS_A_PURCHASED=('IS_A', 'max'),
    
    # IS_B_PURCHASED: whether the basket purchased product B
    IS_B_PURCHASED=('IS_B', 'max'),
    
    # IS_JOINT_PROMOTED: whether the basket contains both promoted A and promoted B
    # Using max() ensures the basket is marked True if any row has IS_PROMO_A = True or IS_PROMO_B = True.
    # A later filter is needed to ensure both A and B are indeed promoted together.
    HAS_PROMO_A=('IS_PROMO_A', 'max'),
    HAS_PROMO_B=('IS_PROMO_B', 'max'),
    
).reset_index()

# 4. Create final treatment and control flags

# JOINT_PURCHASE: whether the basket purchased both A and B
basket_status['IS_A_AND_B_PURCHASED'] = basket_status['IS_A_PURCHASED'] & basket_status['IS_B_PURCHASED']

# JOINT_PROMO_GROUP: baskets where both A and B were promoted simultaneously
basket_status['joint_promo_baskets'] = (basket_status['HAS_PROMO_A'] == True) & (basket_status['HAS_PROMO_B'] == True)


# Non–joint-promotion group: baskets that do not meet the joint-promotion condition 
# (i.e., all baskets minus the joint-promotion group; for simplicity, we take the negation).
basket_status['no_joint_promo_baskets'] = (basket_status['HAS_PROMO_A'] == False) & (basket_status['HAS_PROMO_B'] == False)

basket_status

In [ ]:
# 5. Compute the joint purchase rate P(A ∩ B)
# Formula: P(A ∩ B) = (number of baskets purchasing both A and B) / (total baskets under the condition)

# Group 1 = joint promo
group_joint = basket_status[basket_status['joint_promo_baskets'] == True]
#Group 2 = no joint promo 
group_no_joint = basket_status[basket_status['no_joint_promo_baskets'] == True]

p_ab_joint_promo = group_joint['IS_A_AND_B_PURCHASED'].mean()
p_ab_no_joint_promo = group_no_joint['IS_A_AND_B_PURCHASED'].mean()

p_ab_joint_promo, p_ab_no_joint_promo

In [ ]:
plt.figure(figsize=(6,5))
sns.barplot(
    x=["Joint Promotion", "No Joint Promotion"],
    y=[p_ab_joint_promo, p_ab_no_joint_promo],
    color="#8B5151"
)
plt.title("Combined Purchase Probability: With vs Without Joint Promotion")
plt.ylabel("P(A and B)")
plt.show()


# 5.Customer Segmentation by Purchase Patterns

In [ ]:
# 5-1. Cluster households based on their category purchase combinations

## Clean category names in the product table (trim whitespace to ensure correct grouping)
products['COMMODITY_DESC'] = products['COMMODITY_DESC'].str.strip()

# 2. Merge transaction data with product categories
# Meaning: Link each transaction to its corresponding product category.
household_category = transactions.merge(
    products[['PRODUCT_ID', 'COMMODITY_DESC']],
    on='PRODUCT_ID',
    how='left'
)

# 3. Handle missing/invalid categories (exclude unclassified transactions)
# Meaning: Remove rows where COMMODITY_DESC is empty or invalid (e.g., 'NO COMMODITY DESCRIPTION') to ensure clustering validity.
invalid_categories = ['NO COMMODITY DESCRIPTION', np.nan, ''] 

household_category['invalid_categories'] = household_category['COMMODITY_DESC'].isin(invalid_categories)

household_category = household_category[household_category['invalid_categories'] == False].copy()

household_category = household_category.drop(columns=['invalid_categories'])


household_category

In [ ]:
# classify commodity desc into different categories

category_list = household_category['COMMODITY_DESC'].dropna().unique().tolist()

import re

def classify_category(cat):
    c = cat.upper()

    # 1. PRODUCE (fresh veg and fruit)
    if any(k in c for k in [
        'APPLE', 'FRUIT', 'BERRIES', 'CITRUS', 'MELON', 'VEGETABLE',
        'TOMATO', 'ONION', 'CARROT', 'POTATO', 'SALAD', 'SQUASH',
        'BROCCOLI', 'CAULIFLOWER', 'HERB', 'ORGANICS FRUIT', 'VALUE ADDED VEGETABLE'
    ]):
        return "PRODUCE"

    # 2. MEAT & SEAFOOD
    if any(k in c for k in [
        'BEEF', 'CHICKEN', 'PORK', 'TURKEY', 'LAMB', 'MEAT', 'SEAFOOD', 'FOWL'
    ]):
        return "MEAT_SEAFOOD"

    # 3. DAIRY & EGGS
    if any(k in c for k in [
        'MILK', 'DAIRY', 'YOGURT', 'CHEESE', 'EGG', 'BUTTER', 'SHERBET'
    ]):
        return "DAIRY_EGGS"

    # 4. BAKERY
    if any(k in c for k in [
        'BREAD', 'ROLLS', 'BAKED', 'BAKERY', 'CAKES', 'PIES', 'SWEET GOODS'
    ]):
        return "BAKERY"

    # 5. PANTRY (常溫乾貨)
    if any(k in c for k in [
        'CEREAL', 'NOODLE', 'PASTA', 'BAKING', 'MIX', 'CONDIMENT',
        'SPICE', 'SAUCE', 'CANNED', 'SOUP', 'DRY', 'SUGAR', 'SWEETENER',
        'OIL', 'JELLY', 'JAM', 'RICE', 'POPCORN', 'COFFEE', 'TEA'
    ]):
        return "PANTRY"

    # 6. FROZEN
    if any(k in c for k in [
        'FROZEN', 'FRZN', 'ICE CREAM', 'SHERBTS', 'PIZZA'
    ]):
        return "FROZEN"

    # 7. SNACKS & BEVERAGES
    if any(k in c for k in [
        'SNACK', 'CHIPS', 'CANDY', 'CRACKER', 'SOFT DRINK', 'JUICE',
        'WATER', 'ISOTONIC', 'SODA', 'BEVERAGE', 'NUTS'
    ]):
        return "SNACKS_BEVERAGES"

    # 8. DELI & PREPARED
    if any(k in c for k in [
        'DELI', 'PREPARED', 'SANDWICH', 'HEAT/SERVE', 'SUSHI', 'SALAD BAR'
    ]):
        return "DELI_PREPARED"

    # 9. HOUSEHOLD & PERSONAL CARE
    if any(k in c for k in [
        'TISSUE', 'TOWEL', 'DETERGENT', 'SOAP', 'SHAMPOO', 'HYGIENE',
        'CLEANG', 'BLEACH', 'WRAPS', 'BAGS', 'FOIL', 'PLASTIC HOUSEWARE',
        'COSMETIC', 'DEODORANT', 'SHAVING', 'LAUNDRY'
    ]):
        return "HOUSEHOLD_PERSONAL"

    # 10. SPECIALTY (Baby, Pet, Alcohol, Tobacco, Seasonal, Pharmacy, Misc)
    if any(k in c for k in [
        'BABY', 'INFANT', 'PET', 'DOG', 'CAT', 'WINE', 'LIQUOR', 'BEER',
        'CIG', 'TOBACCO', 'PHARMACY', 'SEASONAL', 'MISC', 'MAGAZINE'
    ]):
        return "SPECIALTY"

    # Fallback: everything else goes to SPECIALTY
    return "SPECIALTY"


CATEGORY_GROUP_DF = pd.DataFrame({
    "COMMODITY_DESC": category_list
})

CATEGORY_GROUP_DF["CATEGORY_GROUP"] = CATEGORY_GROUP_DF["COMMODITY_DESC"].apply(classify_category)

CATEGORY_GROUP_DF



In [ ]:
household_category = household_category.merge(
    CATEGORY_GROUP_DF,
    left_on= 'COMMODITY_DESC',
    right_on= 'COMMODITY_DESC',
    how= 'left'
)

household_category

In [ ]:
household_category[['COMMODITY_DESC', 'CATEGORY_GROUP']].value_counts(dropna=False)

In [ ]:
# --- 2. Create Household x Category Matrix (Pivot) ---

# Aggregate: compute total quantity (QUANTITY) each household (HOUSEHOLD_KEY) purchased in each category (COMMODITY_DESC)
# Meaning: build a multi-dimensional vector representing each household’s purchasing habits.

household_purchase_matrix = household_category.groupby(
    ['household_key','CATEGORY_GROUP'])['QUANTITY'].sum().reset_index(name= 'Total_Quantity')


# Pivot: convert categories into columns and households into rows
# Meaning: reshape the data into the feature-matrix format required for K-Means.
segmentation_matrix = household_purchase_matrix.pivot_table(
    index='household_key', 
    columns='CATEGORY_GROUP', 
    values='Total_Quantity', 
    fill_value=0 # Fill missing values with 0 (if a household never purchased a category, its frequency is 0)
)

segmentation_matrix

In [ ]:
# --- 3. Data Standardization (Z-Score Scaling) ---

# 1. Compute the mean (μ) and standard deviation (σ) for each category
mean_quantity = segmentation_matrix.mean()
std_quantity = segmentation_matrix.std()
std_quantity.replace(0, 1, inplace=True) # 避免除以 0

# 2. Apply Z-score standardization
# Formula: Z = (X - μ) / σ
# Meaning: Scale all features to a comparable range so clustering reflects behavioural patterns rather than raw volumes.
scaled_segmentation = (segmentation_matrix - mean_quantity) / std_quantity

scaled_segmentation

# --- 4. K-means Clustering ---
# Since no elbow method was used to determine K, we set K = 4 for demonstration purposes.
K = 4 

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(scaled_segmentation)

# Retrieve cluster labels
segmentation_matrix['Cluster'] = kmeans.labels_

segmentation_matrix.head(30)


In [ ]:
# visualize the cluster size distribution
fig, ax = plt.subplots()

counts = segmentation_matrix['Cluster'].value_counts()

ax.pie(
    counts.values,           # Values
    labels=counts.index,     # Names for the 4 clusters
    autopct='%1.1f%%',       # Display percentages (recommended)
    startangle=90
)

ax.set_title("Cluster Size Distribution")
plt.show()

In [ ]:
# ---Step 5. Examine the average purchasing pattern of each cluster---

cluster_profile = segmentation_matrix.groupby('Cluster')[segmentation_matrix.columns[:-1]].mean()
cluster_profile

# The total purchase quantity (sum of QUANTITY) for each household within the category group.
# Unit: average QUANTITY purchased.

In [ ]:
# 5-2. Identify distinct shopping missions (e.g., “weekly stock-up,” “quick meals,” “entertainment”)

# --- Extract Cluster Centers ---
# Meaning: The K-Means model stores the cluster centers in the `.cluster_centers_` attribute.
# These centers are the mean vectors in the scaled_df feature space.
scaled_centers = kmeans.cluster_centers_

# Required feature columns (excluding the cluster label column)
feature_cols = segmentation_matrix.columns.drop('Cluster')

# --- Inverse Scaling ---
# Meaning: Convert the cluster centers from the Z-score space (mean=0, std=1)
# back to the original Total_Quantity scale for business interpretation.
# Formula: X = Z * σ + μ
# --- Inverse scaling (back to original units) ---
cluster_centers_original = scaled_centers * std_quantity[feature_cols].values + mean_quantity[feature_cols].values


# Convert results to a DataFrame with original column names for easy inspection
cluster_profile = pd.DataFrame(
    cluster_centers_original, 
    columns=feature_cols
)

cluster_profile.index = [f"Cluster {i}" for i in range(K)]

# cluster_profile['Cluster'] = range(K)
# cluster_profile = cluster_profile.set_index('Cluster')
cluster_profile

In [ ]:
# Z-score version of the cluster profile (best for identifying shopping missions)
cluster_profile_zscore = (cluster_profile - cluster_profile.mean()) / cluster_profile.std()
cluster_profile_zscore

In [ ]:
# visualize the distinct shopping missions of each cluster
plt.figure(figsize=(14, 8))
sns.heatmap(
    cluster_profile_zscore,
    cmap="coolwarm",          # Red = above average, Blue = below average
    center=0,
    annot=False,              # Do not annotate if there are too many categories
    linewidths=0.3,
    cbar_kws={'label': 'Z-Score'}
)

plt.title("Cluster Profile Heatmap (Z-Score Normalized)", fontsize=14, pad=12)
plt.xlabel("Category Group")
plt.ylabel("Cluster")
plt.show()

In [ ]:
pair_filtered.columns

In [ ]:
# If old CATEGORY_A/B columns exist, drop them first to avoid duplication
pair_filtered = pair_filtered.loc[:, ~pair_filtered.columns.duplicated()]
pair_filtered = pair_filtered.drop(columns=[col for col in pair_filtered.columns 
                                            if col in ['CATEGORY_A', 'CATEGORY_B']], errors='ignore')
pair_filtered = pair_filtered.drop(columns=[col for col in pair_filtered.columns 
                                            if col in ['CATEGORY_GROUP_A', 'CATEGORY_GROUP_B']], errors='ignore')



In [ ]:
# 5-3. For each segment, identify most common product affinity patterns

# =============================
# Merge PRODUCT_ID_A / B with the original product category info
# =============================

# Merge category info for A
high_lift_pairs = high_lift_pairs.merge(
    products[['PRODUCT_ID', 'COMMODITY_DESC']],
    left_on= 'PRODUCT_ID_A',
    right_on= 'PRODUCT_ID',
    how= 'left',
    suffixes=('', '_A')
).rename(columns={'COMMODITY_DESC': 'CATEGORY_A'}).drop(columns=['PRODUCT_ID'])


# Merge category info for B
high_lift_pairs = high_lift_pairs.merge(
    products[['PRODUCT_ID', 'COMMODITY_DESC']],
    left_on= 'PRODUCT_ID_B',
    right_on= 'PRODUCT_ID',
    how= 'left',
    suffixes=('', '_A')
).rename(columns={'COMMODITY_DESC': 'CATEGORY_B'}).drop(columns=['PRODUCT_ID'])


# Merge the category group (the 10 grouped categories)
high_lift_pairs = high_lift_pairs.merge(
    CATEGORY_GROUP_DF,
    left_on='CATEGORY_A',
    right_on='COMMODITY_DESC',
    how='left'
).rename(columns={'CATEGORY_GROUP': 'CATEGORY_GROUP_A'}).drop(columns=['COMMODITY_DESC'])

# Merge the category group (the 10 grouped categories)
high_lift_pairs = high_lift_pairs.merge(
    CATEGORY_GROUP_DF,
    left_on='CATEGORY_B',
    right_on='COMMODITY_DESC',
    how='left'
).rename(columns={'CATEGORY_GROUP': 'CATEGORY_GROUP_B'}).drop(columns=['COMMODITY_DESC'])

high_lift_pairs



In [ ]:
# find top three categories of each cluster
cluster_top_categories = {}

for cluster in cluster_profile_zscore.index:
    row = cluster_profile_zscore.loc[cluster]
    cluster_top_categories[cluster] = row.nlargest(3).index.tolist()

cluster_top_categories

In [ ]:
pair_filtered

In [ ]:
# Identify top product affinity patterns per cluster

cluster_affinity_products = {}

for cluster, top_groups in cluster_top_categories.items():
    
    # Filter pairs: both A and B must belong to the cluster's top category groups
    subset = high_lift_pairs[
        high_lift_pairs['CATEGORY_GROUP_A'].isin(top_groups) &
        high_lift_pairs['CATEGORY_GROUP_B'].isin(top_groups)
    ].copy()
    
    # Sort by lift
    top_pairs = subset.sort_values('Lift', ascending=False).head(10)
    
    cluster_affinity_products[cluster] = top_pairs

result_df_list = []

for cluster, df in cluster_affinity_products.items():
    temp = df.copy()
    temp['Cluster'] = cluster
    result_df_list.append(temp)

cluster_affinity_products_df = pd.concat(result_df_list, ignore_index=True)
cluster_affinity_products_df


In [ ]:
high_lift_pairs

In [ ]:
# 5-4. Develop segment-specific cross-sell recommendations

# why use high_lift_pair:
#   Keep only the pairs that are truly meaningful (high-affinity item pairs)

# What Part 5-4 is doing:
#   For each customer segment (Cluster), identify the most recommendable item pairs (A → B).
#   Rank each pair using a score: Score = Lift × Confidence.
#   Output: a mapping of Cluster → Top 10 recommended pairs.
#   In other words, generate cross-sell strategies tailored to each segment.

cluster_cross_sell = []

for cluster, top_groups in cluster_top_categories.items():

    subset = high_lift_pairs[
        # For this customer segment, only consider pairs related to the top categories they purchase most frequently.
        high_lift_pairs['CATEGORY_GROUP_A'].isin(top_groups) &
        high_lift_pairs['CATEGORY_GROUP_B'].isin(top_groups)
    ].copy()

    # Lift measures the strength of the effect (co-occurrence uplift).
# Confidence measures the stability / frequency of the rule.
# Score = Lift × Confidence → When both are high:
#   The pair is not just a random coincidence;
#   A high proportion of A-buyers also purchase B;
#   The recommendation is both commercially meaningful and statistically supported.
    subset['CROSS_SELL_SCORE'] = subset['Lift'] * subset['Confidence_A_B']
    subset['Cluster'] = cluster

    cluster_cross_sell.append(subset)


# Merge everything into a final consolidated table (final output DataFrame)
cluster_cross_sell_df = pd.concat(cluster_cross_sell, ignore_index=True)

cluster_cross_sell_df



# 6. Recommendation Engine Prototype

In [ ]:
# 6-1. Build simple recommendation logic:
# ▪ Given products in current basket, predict next likely purchases
# ▪ Rank recommendations by lift × confidence

# Cross-sell Score

rules_df = high_lift_pairs.copy()

# Create the "recommendation strength score": Score = Lift × Confidence(A→B)
# Multiplying the two helps identify rules that are both strong (effect) and reliable (stability).
rules_df['CROSS_SELL_SCORE'] = rules_df['Lift'] * rules_df['Confidence_A_B']

# To speed up later lookups, group the rules by PRODUCT_ID_A and store them in a dictionary
rules_by_A = {pid: group for pid, group in rules_df.groupby('PRODUCT_ID_A')}

def recommend_from_basket(basket_products, rules_by_A, top_k=10):
    """
    basket_products: list / set of PRODUCT_ID，代表目前籃子裡的商品
    rules_by_A: 前面建立好的 {PRODUCT_ID_A: 對應規則 DataFrame}
    top_k: 想要幾個推薦
    """
    
    # 1. Use a dict to accumulate the total score of each candidate product B
    candidate_scores = {}

    # 2. For each item in the basket as A, retrieve all A→B rules
    for pid in basket_products:
        if pid not in rules_by_A:
            continue  # No rules found for this product → skip
        
        rules_for_A = rules_by_A[pid]  # All rules where A = pid
        
        # 3. Accumulate the scores for all candidate products B
        for _, row in rules_for_A.iterrows():
            b = row['PRODUCT_ID_B']
            
            # Do not recommend items that are already in the basket
            if b in basket_products:
                continue
            
            score = row['CROSS_SELL_SCORE']
            
            # If multiple A's recommend the same B, accumulate their scores (signal stacking).
            # If B already has a score → add the new score; if not → initialize at 0 and then add the score.
            candidate_scores[b] = candidate_scores.get(b, 0) + score

    # 4. If no candidate products exist, return an empty DataFrame
    if not candidate_scores:
        return pd.DataFrame(columns=['PRODUCT_ID', 'SCORE'])
    
    # 5. Convert the dict to a DataFrame and sort it
    recommend_df = (
        pd.DataFrame(
            [{'PRODUCT_ID': pid, 'SCORE': s} for pid, s in candidate_scores.items()]
        )
        .sort_values('SCORE', ascending=False)
        .head(top_k)
    )
    
    # 6. Add product names and categories to improve interpretability
    recommend_df = recommend_df.merge(
        products[['PRODUCT_ID', 'COMMODITY_DESC']],
        on='PRODUCT_ID',
        how='left'
    )
    
    return recommend_df



In [ ]:
# 6-2. Test recommendation accuracy on held-out transaction data

# Build a basket-level DataFrame: one row per BASKET_ID with its list of products

basket_df = (
    transactions
    .groupby('BASKET_ID')['PRODUCT_ID']
    .apply(list)          # Each BASKET_ID is mapped to its product list
    .reset_index()
    .rename(columns={'PRODUCT_ID': 'PRODUCT_LIST'})
)

basket_df.head()


In [ ]:
# 6-4-1. Perform a simple train/test split (using baskets as the unit)
train_baskets, test_baskets = train_test_split(
    basket_df,
    test_size=0.2,
    random_state=42
)

# 6-4-2. Define the evaluation function: Hit Rate@K
def evaluate_recommender(baskets_df, rules_by_A, top_k=10, max_samples=5000):
    """
    baskets_df: the basket DataFrame to evaluate (typically test_baskets)
    rules_by_A: the rules dictionary we created earlier
    top_k: length of the recommendation list (sorted by Lift × Confidence as required)
    max_samples: maximum number of baskets to sample for evaluation to avoid slow runtime
    """
    
    hits = 0      # Count of correct hits
    total = 0     # Number of valid test cases
    
    # Keep only baskets with at least 2 distinct products; otherwise we cannot split into input/target
    more_than_two_prod = [len(set(p)) >= 2 for p in baskets_df["PRODUCT_LIST"]]
    eligible = baskets_df[more_than_two_prod]
    
    # Randomly sample a subset of baskets for evaluation (to avoid heavy computation)
    sample = eligible.sample(
        n=min(max_samples, len(eligible)),
        random_state=42
    )
    
    for _, row in sample.iterrows():
        products = list(dict.fromkeys(row['PRODUCT_LIST']))  # Remove duplicates while preserving order
        if len(products) < 2:
            continue
        
        # Use the last item as the prediction target, and treat all preceding items as observed inputs
        target = products[-1]
        observed = products[:-1]
        
        # Use observed items to generate recommendations
        rec_df = recommend_from_basket(observed, rules_by_A, top_k=top_k)
        
        # Count as hit if target is in the recommendations
        if target in rec_df['PRODUCT_ID'].values:
            hits += 1
        
        total += 1
    
    hit_rate = hits / total if total > 0 else 0
    return hit_rate

# Compute the actual Hit Rate@10
hit_rate_at_10 = evaluate_recommender(test_baskets, rules_by_A, top_k=10)
hit_rate_at_10
